In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd

full_path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(full_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
for col in df.columns:
  df[col] = df[col].fillna(df[col].median())

In [ ]:
# Task 2: Write your code here:
df.drop_duplicates(inplace=True)

In [ ]:
# Task 3: Write your code here:
# There are no catagories

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
# Have done splitting early
from sklearn.model_selection import train_test_split
X = df.drop('Target', axis=1)
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
plt.hist(y.dropna(), bins=50, edgecolor='black')
# There is some imbalance, so stratified split is needed (done already above)

In [ ]:
# Task 1: Write your code here:
#done above

In [ ]:
# Task 2,3,4,5: Write your code here:
# %pip install catboost # Needed to run it once
from catboost import CatBoostClassifier
from sklearn.model_selection import KFold
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

kfold = KFold(n_splits=5, shuffle=True, random_state=42) # I just noticed that I was using booth kfold and train_test_split in two labs D:<

model = CatBoostClassifier(
      verbose=0,
      n_estimators=200,
      max_depth=4
  )

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  {mae_scores.mean():,.2f}")
print(f"RMSE: {rmse_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
feature_importance.max()
# Feature S_9 is the most important!


In [ ]:
# Task Bonus: Write your code here: